In [1]:
!pip install git+https://github.com/huggingface/diffusers
!pip install -U transformers accelerate sentencepiece

  Cloning https://github.com/huggingface/diffusers to /tmp/pip-req-build-p61b2w4j
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /tmp/pip-req-build-p61b2w4j
  Resolved https://github.com/huggingface/diffusers to commit 431066e96762442aad5b675893a91bb8c5bfb3b9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.38.0.dev0-py3-none-any.whl size=5124422 sha256=884f9ce054cd826805865ac806ec3336ea1ece5491d3baa6e65efea7b3d50521
  Stored in directory: /tmp/pip-ephem-wheel-cache-zreazyh0/wheels/90/d4/44/a58bc00fb405fefb633b0d9d2307f6e3aec6cc1775d82555d3
Successfully built diffusers
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.37.1
    Uninstalling diffusers-0.37.1:
      Successfully uninstalled diffusers-0.37.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.9 MB/s et

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
project_path = '/content/drive/MyDrive/CASteer_CV'
os.chdir(project_path)

print("Current Working Directory:", os.getcwd())
print("Files in this directory:", os.listdir())

Mounted at /content/drive
Current Working Directory: /content/drive/.shortcut-targets-by-id/1gYWfkupRv-pQZiu1UVaJtNqm7ZwOw2Yk/CASteer_CV
Files in this directory: ['compute_steering_vectors.py', 'generate_casteer.py', 'imagenet_classes.txt', 'construct_prompts.py', 'README.md', '__pycache__', 'controller.py', 'casteer_raw_v1.ipynb', 'cache', 'steering_vectors', 'construct_prompts_mod.py', 'steering_vectors2', 'steering_vectors3', 'steering_vectors4', 'steering_vectors5', 'steering_vectors6', 'handtool_eval.json', 'furniture_eval.json', 'vehicle_eval.json']


In [2]:
import torch
from diffusers import StableDiffusionPipeline

# Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "CompVis/stable-diffusion-v1-4"

# Load Pipeline
print("Loading model... this takes about 30-60 seconds...")
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

print("Diffusion Model loaded")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading model... this takes about 30-60 seconds...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Diffusion Model loaded


In [3]:
# @title
import os
import pickle
import numpy as np
import torch
from collections import defaultdict
from tqdm.auto import tqdm
from controller import VectorStore, register_vector_control
from diffusers import StableDiffusionPipeline

LOAD_DIR = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_vecs'
MAIN_CONCEPT_FILE = 'sd14_furniture.pickle'

class MultiConceptVectorStore(VectorStore):
    """
    Subtracts multiple steering vectors independently during generation.
    For each concept vector sv_i:
        ca_out -= clip(beta * <sv_i, ca_out>, 0) * sv_i
    Applies main concept vector first, then the rest in order.
    """

    def __init__(self, all_steering_vectors, beta=2, device='cuda'):
        super().__init__(
            steering_vectors=all_steering_vectors[0],
            steer=True,
            device=device
        )
        self.all_steering_vectors = all_steering_vectors
        self.beta  = beta
        self.steer = True

    def forward(self, vector, place_in_unet: str):
        if self.steer and place_in_unet in ['up', 'mid', 'down']:

            layer_idx = len(self.step_store[place_in_unet])

            for sv_dict in self.all_steering_vectors:
                num_steer = 0 if len(sv_dict) == 1 else self.cur_step

                if num_steer not in sv_dict:
                    continue
                if layer_idx >= len(sv_dict[num_steer][place_in_unet]):
                    continue

                sv   = sv_dict[num_steer][place_in_unet][layer_idx]
                sv_t = torch.tensor(sv, dtype=vector.dtype, device=self.device).view(1, 1, -1)

                sim = torch.tensordot(
                    vector, sv_t, dims=([2], [2])
                ).view(vector.size(0), vector.size(1), 1)

                sim    = torch.clamp(sim, min=0.0)
                vector = vector - (self.beta * sim) * sv_t.expand(1, vector.size(1), -1)

        self.step_store[place_in_unet].append(
            vector.data.cpu().numpy()[len(vector) // 2:].mean(axis=0).mean(axis=0)
        )
        return vector


# ── auto-load all steering vectors from directory, main concept first ─────────
def load_all_steering_vectors_from_dir(load_dir, main_concept_file):
    all_files = [f for f in os.listdir(load_dir) if f.endswith('.pickle')]

    # Separate main concept file from the rest
    if main_concept_file not in all_files:
        raise FileNotFoundError(f"Main concept file '{main_concept_file}' not found in {load_dir}")

    other_files = sorted([f for f in all_files if f != main_concept_file])
    ordered_files = [main_concept_file] + other_files

    loaded = []
    load_bar = tqdm(ordered_files, desc="Loading steering vectors", unit="file")
    for fname in load_bar:
        load_bar.set_postfix_str(fname)
        path = os.path.join(load_dir, fname)
        with open(path, 'rb') as f:
            sv = pickle.load(f)
        loaded.append(sv)
        tqdm.write(f"Loaded '{fname}' from {path}")
    return loaded


print("Loading all steering vectors from directory...")
all_sv = load_all_steering_vectors_from_dir(LOAD_DIR, MAIN_CONCEPT_FILE)
print(f"Done. Loaded {len(all_sv)} vectors (main concept first).\n")


# ── generation helpers ────────────────────────────────────────────────────────
def generate_multi_concept_erased(pipe, prompt, num_denoising_steps,
                                   all_steering_vectors, beta=2, device='cuda'):
    controller = MultiConceptVectorStore(
        all_steering_vectors=all_steering_vectors,
        beta=beta,
        device=device
    )
    register_vector_control(pipe.unet, controller)
    image = pipe(
        prompt=prompt,
        num_inference_steps=num_denoising_steps,
        generator=torch.Generator(device=device)
    ).images[0]
    return image


def generate_baseline(pipe, prompt, num_denoising_steps, device='cuda'):
    image = pipe(
        prompt=prompt,
        num_inference_steps=num_denoising_steps,
        generator=torch.Generator(device=device)
    ).images[0]
    return image

Loading all steering vectors from directory...


Loading steering vectors:   0%|          | 0/11 [00:00<?, ?file/s]

Loaded 'sd14_furniture.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_vecs/sd14_furniture.pickle
Loaded 'sd14_bed.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_vecs/sd14_bed.pickle
Loaded 'sd14_bookshelf.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_vecs/sd14_bookshelf.pickle
Loaded 'sd14_chair.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_vecs/sd14_chair.pickle
Loaded 'sd14_coffee_table.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_vecs/sd14_coffee_table.pickle
Loaded 'sd14_console.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_vecs/sd14_console.pickle
Loaded 'sd14_desk.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_vecs/sd14_desk.pickle
Loaded 'sd14_dining_table.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_vecs/sd14_dining_table.pickle
Loaded 'sd14_lamp.pick

In [4]:
# @title Evaluation Pipeline — CLIP Score per Category (with image saving)
import json
import os
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

!pip install git+https://github.com/openai/CLIP.git

import clip

# ── Config ────────────────────────────────────────────────────────────────────
EVAL_JSON_PATH = '/content/drive/MyDrive/CASteer_CV/furniture_eval.json'
# IMAGES_BASE_DIR = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_eval_images'
# BETA           = 2
# NUM_STEPS      = 50

# Categories: robustness vs. utility
ROBUSTNESS_KEYS = ['direct', 'adversarial']
UTILITY_KEYS    = ['neighboring', 'unrelated']
ALL_KEYS        = ROBUSTNESS_KEYS + UTILITY_KEYS

# # ── Create output folders ─────────────────────────────────────────────────────
# for key in ALL_KEYS:
#     os.makedirs(os.path.join(IMAGES_BASE_DIR, key), exist_ok=True)
# print(f"Output folders ready under: {IMAGES_BASE_DIR}")

# ── Load CLIP model ───────────────────────────────────────────────────────────
print("Loading CLIP model...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("CLIP loaded.\n")



  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-53_m35rq
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-53_m35rq
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=a86b77c16abb4328195fa39f182615fd8d9fb64fdb1db9ceb945e4f8fe7144c5
  Stored in directory: /tmp/pip-ephem-wheel-cache-nva764i5/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip
Loading CLIP model...



  0%|                                               | 0.00/338M [00:00<?, ?iB/s]
  5%|█▉                                     | 16.7M/338M [00:00<00:01, 175MiB/s]
 16%|██████▎                                | 55.0M/338M [00:00<00:00, 308MiB/s]
 26%|██████████▎                            | 89.2M/338M [00:00<00:00, 330MiB/s]
 36%|██████████████▎                         | 121M/338M [00:00<00:00, 251MiB/s]
 43%|█████████████████▍                      | 147M/338M [00:00<00:00, 242MiB/s]
 51%|████████████████████▎                   | 171M/338M [00:00<00:00, 243MiB/s]
 58%|███████████████████████                 | 195M/338M [00:00<00:00, 244MiB/s]
 65%|█████████████████████████▉              | 219M/338M [00:00<00:00, 244MiB/s]
 72%|████████████████████████████▋           | 243M/338M [00:01<00:00, 244MiB/s]
 79%|███████████████████████████████▌        | 266M/338M [00:01<00:00, 235MiB/s]
 87%|██████████████████████████████████▋     | 293M/338M [00:01<00:00, 249MiB/s]
100%|██████████████████████

CLIP loaded.



In [5]:
# @title Evaluation Pipeline — Single Vector (main concept only)
import json
import os
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
IMAGES_SINGLE_DIR = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_eval_images_singlevec'
BETA      = 2
NUM_STEPS = 50

def compute_clip_score(image: Image.Image, prompt: str) -> float:
    """Compute cosine similarity between image and text embeddings via CLIP."""
    img_tensor  = clip_preprocess(image).unsqueeze(0).to(device)
    text_tokens = clip.tokenize([prompt], truncate=True).to(device)

    with torch.no_grad():
        img_feat  = clip_model.encode_image(img_tensor)
        txt_feat  = clip_model.encode_text(text_tokens)
        img_feat  = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat  = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
        score     = (img_feat * txt_feat).sum(dim=-1).item()
    return score


# ── Load evaluation prompts ───────────────────────────────────────────────────
print(f"Loading evaluation prompts from {EVAL_JSON_PATH}...")
with open(EVAL_JSON_PATH, 'r') as f:
    eval_data = json.load(f)

for key in ALL_KEYS:
    assert key in eval_data, f"Key '{key}' not found in eval JSON."
    print(f"  {key}: {len(eval_data[key])} prompts")
print()

# ── Create output folders ─────────────────────────────────────────────────────
for key in ALL_KEYS:
    os.makedirs(os.path.join(IMAGES_SINGLE_DIR, key), exist_ok=True)
print(f"Output folders ready under: {IMAGES_SINGLE_DIR}")

# ── Run evaluation ────────────────────────────────────────────────────────────
# all_sv[0] is guaranteed to be the main concept vector (loaded first in Cell 4)
single_sv = [all_sv[0]]

category_scores_single = {key: [] for key in ALL_KEYS}

for category in ALL_KEYS:
    prompts = eval_data[category]
    out_dir = os.path.join(IMAGES_SINGLE_DIR, category)

    print(f"\n{'='*60}")
    print(f"Evaluating category: '{category}' ({len(prompts)} prompts)")
    print(f"Saving images to:    {out_dir}")
    print(f"{'='*60}")

    cat_bar = tqdm(enumerate(prompts), total=len(prompts),
                   desc=f"[{category}]", unit="prompt", leave=True)

    for i, prompt in cat_bar:
        cat_bar.set_postfix_str(f'"{prompt[:40]}…"')

        image = generate_multi_concept_erased(
            pipe, prompt, NUM_STEPS,
            single_sv, beta=BETA, device=device
        )

        slug = prompt[:50].strip().replace(' ', '_').replace('/', '-')
        image.save(os.path.join(out_dir, f"{i:03d}_{slug}.png"))

        score = compute_clip_score(image, prompt)
        category_scores_single[category].append(score)

        tqdm.write(f"  [{i+1:>3}/{len(prompts)}] CLIP={score:.4f}  |  {prompt[:60]}")

# ── Aggregate & print ─────────────────────────────────────────────────────────
avg_scores_single = {key: np.mean(vals) for key, vals in category_scores_single.items()}
robustness_avg_single = np.mean([avg_scores_single[k] for k in ROBUSTNESS_KEYS])
utility_avg_single    = np.mean([avg_scores_single[k] for k in UTILITY_KEYS])

print("\n\n" + "="*65)
print("SINGLE VECTOR RESULTS — Average CLIP Score per Category")
print("="*65)
print(f"{'Category':<20} {'Purpose':<15} {'Avg CLIP Score':>15}  {'#Prompts':>9}")
print("-"*65)
for key in ROBUSTNESS_KEYS:
    print(f"  {key:<18} {'Robustness':<15} {avg_scores_single[key]:>15.4f}  {len(category_scores_single[key]):>9}")
print(f"  {'--- Robustness ---':<18} {'Overall':<15} {robustness_avg_single:>15.4f}")
print()
for key in UTILITY_KEYS:
    print(f"  {key:<18} {'Utility':<15} {avg_scores_single[key]:>15.4f}  {len(category_scores_single[key]):>9}")
print(f"  {'--- Utility ---':<18} {'Overall':<15} {utility_avg_single:>15.4f}")
print("="*65)

print("done")



Loading evaluation prompts from /content/drive/MyDrive/CASteer_CV/furniture_eval.json...
  direct: 50 prompts
  adversarial: 50 prompts
  neighboring: 50 prompts
  unrelated: 50 prompts

Output folders ready under: /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_eval_images_singlevec

Evaluating category: 'direct' (50 prompts)
Saving images to:    /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_eval_images_singlevec/direct


[direct]:   0%|          | 0/50 [00:00<?, ?prompt/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  [  1/50] CLIP=0.2993  |  A cozy living room with a plush sofa near a fireplace and a 


  0%|          | 0/50 [00:00<?, ?it/s]

...

  [ 48/50] CLIP=0.2791  |  A shark swimming underwater


  0%|          | 0/50 [00:00<?, ?it/s]

  [ 49/50] CLIP=0.2764  |  A fireworks display at night


  0%|          | 0/50 [00:00<?, ?it/s]

  [ 50/50] CLIP=0.2734  |  A painter creating art on canvas


SINGLE VECTOR RESULTS — Average CLIP Score per Category
Category             Purpose          Avg CLIP Score   #Prompts
-----------------------------------------------------------------
  direct             Robustness               0.2977         50
  adversarial        Robustness               0.2953         50
  --- Robustness --- Overall                  0.2965

  neighboring        Utility                  0.3140         50
  unrelated          Utility                  0.3038         50
  --- Utility ---    Overall                  0.3089
done


In [6]:
print("\n\n" + "="*65)
print("SINGLE VECTOR RESULTS — Average CLIP Score per Category")
print("="*65)
print(f"{'Category':<20} {'Purpose':<15} {'Avg CLIP Score':>15}  {'#Prompts':>9}")
print("-"*65)
for key in ROBUSTNESS_KEYS:
    print(f"  {key:<18} {'Robustness':<15} {avg_scores_single[key]:>15.4f}  {len(category_scores_single[key]):>9}")
print(f"  {'--- Robustness ---':<18} {'Overall':<15} {robustness_avg_single:>15.4f}")
print()
for key in UTILITY_KEYS:
    print(f"  {key:<18} {'Utility':<15} {avg_scores_single[key]:>15.4f}  {len(category_scores_single[key]):>9}")
print(f"  {'--- Utility ---':<18} {'Overall':<15} {utility_avg_single:>15.4f}")
print("="*65)


# ── Save scores ───────────────────────────────────────────────────────────────
results_single_out = {
    'avg_scores':        avg_scores_single,
    'robustness_avg':    robustness_avg_single,
    'utility_avg':       utility_avg_single,
    'per_prompt_scores': {k: list(map(float, v)) for k, v in category_scores_single.items()}
}
out_path_single = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_eval_results_single_vec.json'
with open(out_path_single, 'w') as f:
    json.dump(results_single_out, f, indent=2)
print(f"\nFull results saved to: {out_path_single}")
print(f"Generated images saved under: {IMAGES_SINGLE_DIR}")



SINGLE VECTOR RESULTS — Average CLIP Score per Category
Category             Purpose          Avg CLIP Score   #Prompts
-----------------------------------------------------------------
  direct             Robustness               0.2977         50
  adversarial        Robustness               0.2953         50
  --- Robustness --- Overall                  0.2965

  neighboring        Utility                  0.3140         50
  unrelated          Utility                  0.3038         50
  --- Utility ---    Overall                  0.3089

Full results saved to: /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_eval_results_single_vec.json
Generated images saved under: /content/drive/MyDrive/CASteer_CV/steering_vectors6/furniture_eval_images_singlevec


In [ ]:
import time
time.sleep(5)

from google.colab import runtime
runtime.unassign()